In [1]:
import sys
sys.path.append('..')
import numpy as np

from util.plot import start_end_subplot

from model import models, networks
from model.models import EDMPrecond,FMCond
from model.networks import MLP,MLP_dd
import trimesh

from util.mesh_utils import *

device='cuda:0'
torch.cuda.set_device(device)

PLOT=True

tinycudann is available.


In [ ]:
NUM_POINTS_INFERENCE=10000
NUM_STEP=64

SOURCE="tr_reg_080"
TARGET="tr_reg_093"

In [ ]:
mesh1= normalize_mesh(trimesh.load('../../Matching-Flowing-Distributions/data/tr_reg_080.off'))
mesh2= normalize_mesh(trimesh.load('../../Matching-Flowing-Distributions/data/tr_reg_093.off'))
model = FMCond(channels=3 , network=MLP().to(device))   
model.to(device)
model.load_state_dict(torch.load('./../output/'+SOURCE+'/checkpoint-9999.pth', map_location=device,weights_only=False)['model'], strict=True)


model2 = FMCond(channels=3 , network=MLP().to(device))   
model2.to(device)
model2.load_state_dict(torch.load('./../output/'+TARGET+'/checkpoint-9999.pth', map_location=device,weights_only=False)['model'], strict=True)

model.eval()
model2.eval()


In [ ]:
v1= torch.tensor(mesh1.vertices).to(torch.float32).to(device)
v2= torch.tensor(mesh2.vertices).to(torch.float32).to(device)
v1_pullback = model.inverse(samples=v1, num_steps= NUM_STEP)
v1_pullback = v1_pullback.to(torch.float32)
v2_pullback= model2.inverse(samples=v2, num_steps= NUM_STEP)
v2_pullback = v2_pullback.to(torch.float32)

if PLOT:
    start_end_subplot(v1.cpu().detach(),v1_pullback.cpu().detach(),show=True)
    start_end_subplot(v2.cpu().detach(),v2_pullback.cpu().detach(),show=True)

# Identity

In [ ]:
with torch.no_grad():
    noise2 = model.inverse(samples=v1, num_steps= NUM_STEP)
    sample2 = model2.sample(noise=noise2, num_steps=NUM_STEP)
    
print(torch.mean((sample2-v2)**2))
if PLOT:
    start_end_subplot(v1.cpu(),sample2.cpu() ,show=True)


# Regid Rotation

In [ ]:
from util.matching_utils import *

In [ ]:
# WITH fm
parameters = torch.randn(3, requires_grad=True, device=device)

optimizer = torch.optim.Adam([parameters], lr=0.01)

n_epoch=1000
for step in range(n_epoch):  
    optimizer.zero_grad() 

    Rot = euler_to_SO3(parameters)

    # Apply the rotation and compute the loss
    pullback_rotated = v1_pullback @ Rot

    loss = torch.mean((pullback_rotated - v2_pullback) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

sol=model2.sample(noise=v1_pullback@Rot, num_steps=64,enable_grad=False)
print(torch.mean((sol - v2) ** 2))

if PLOT:
    start_end_subplot(v2.cpu(),sol.cpu(),show=True)
    start_end_subplot(v1_pullback.cpu(),(v1_pullback@Rot).detach().cpu(),show=True)


In [ ]:
# Without FM
parameters = torch.randn(3, requires_grad=True, device=device)
optimizer = torch.optim.Adam([parameters], lr=0.01)

n_epoch=1000
for step in range(n_epoch):  
    optimizer.zero_grad()  

    Rot = euler_to_SO3(parameters)

    solution_rotated = v1 @ Rot
    loss = torch.mean((solution_rotated - v2) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

solution_rotated=v1@Rot
print(torch.mean((solution_rotated - v2) ** 2))

if PLOT:
    start_end_subplot(v2.detach().cpu(),solution_rotated.detach().cpu(),show=True)


# Non Linear Map

In [ ]:
# with FM
from util.matching_utils import *
model_nn=MLP_xyz().to(device)

optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.01)

n_epoch=1000
for step in range(n_epoch):  # Number of optimization steps
    optimizer.zero_grad()  # Clear previous gradients

    f_v1_pullback = model_nn(v1_pullback)
    loss = torch.mean((f_v1_pullback - v2_pullback) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")
    
sol=model2.sample(noise=f_v1_pullback, num_steps=64,enable_grad=False)
print(torch.mean((sol - v2) ** 2))

if PLOT:
    start_end_subplot(v1_pullback.cpu(), f_v1_pullback.detach().cpu(),show=True)
    start_end_subplot(v2.cpu(), sol.detach().cpu(),show=True)

In [ ]:
# without FM
model_nn=MLP_xyz().to(device)

optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.01)

n_epoch=1000
for step in range(n_epoch):
    optimizer.zero_grad() 

    f_v1 = model_nn(v1)
    loss = torch.mean((f_v1 - v2) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")
        
print(torch.mean((f_v1 - v2) ** 2))

if PLOT:
    start_end_subplot(v2.cpu(), f_v1.detach().cpu(),show=True)


# Invertible Map

In [ ]:
import sys
sys.path.append('../implicit-arap/')
from iarap.model.nn.invertible import InvertibleMLP3D,InvertibleMLP3DConfig

In [ ]:
# with FM
config=InvertibleMLP3DConfig()
config.activation = 'ReLU'
config.act_defaults = {}
model_nn=InvertibleMLP3D(config).to(device)

optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.0001)

n_epoch=1000
for step in range(n_epoch):  # Number of optimization steps
    optimizer.zero_grad()  # Clear previous gradients

    f_v1_pullback = model_nn(v1_pullback)[0]
    loss = torch.mean((f_v1_pullback - v2_pullback) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")
    
    
f_v1_pullback = model_nn(v1_pullback)[0]
sol=model2.sample(noise=f_v1_pullback, num_steps=64,enable_grad=False)
print(torch.mean((sol - v2) ** 2))

if PLOT:
    start_end_subplot(v1_pullback.cpu(), f_v1_pullback.detach().cpu(),show=True)
    start_end_subplot(v2.cpu(), sol.detach().cpu(),show=True)

In [ ]:
# without FM
config=InvertibleMLP3DConfig()
config.activation = 'ReLU'
config.act_defaults = {}
model_nn=InvertibleMLP3D(config).to(device)

optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.0001)

n_epoch=1000
for step in range(n_epoch):  # Number of optimization steps
    optimizer.zero_grad()  # Clear previous gradients

    f_v1 = model_nn(v1)[0]
    loss = torch.mean((f_v1 - v2) ** 2)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")
    
    
f_v1 = model_nn(v1)[0]
print(torch.mean((f_v1 - v2) ** 2))

if PLOT:
    start_end_subplot(v2.cpu(), f_v1.detach().cpu(),show=True)

# Flows (Straight)

In [ ]:
from models import FMCond
from models import MLP, Network
from flow_matching.path.scheduler import CondOTScheduler, LinearVPScheduler,VPScheduler

flowB= FMCond(channels=3 , network=MLP().to(device))
optimizer = torch.optim.Adam(flowB.parameters(), lr=0.01)

path=AffineProbPath(scheduler=CondOTScheduler())

n_epoch=1000
for step in range(n_epoch): 
    optimizer.zero_grad()  

    t=torch.rand(v1_pullback.shape[0], device=device)

    path_sample = path.sample(t=t, x_0=v1_pullback, x_1=v2_pullback)
    v_t=flowB(path_sample.x_t, path_sample.t)
    loss = torch.mean((v_t - path_sample.dx_t) ** 2)
    
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

sol=model2.sample(noise=flowB.sample(noise=v1_pullback,num_steps=64), num_steps=64)
print(torch.mean((sol - v2) ** 2))
if PLOT:
    start_end_subplot(v1_pullback.cpu(), flowB.sample(noise=v1_pullback,num_steps=64).detach().cpu()  ,show=True)
    start_end_subplot(v2.cpu(), sol.detach().cpu(),show=True)

In [ ]:
from models import FMCond
from models import MLP, Network
from flow_matching.path.scheduler import CondOTScheduler, LinearVPScheduler,VPScheduler

flowB= FMCond(channels=3 , network=MLP(input_dim=53,output_dim=53).to(device))
optimizer = torch.optim.Adam(flowB.parameters(), lr=0.01)

path=AffineProbPath(scheduler=CondOTScheduler())

n_epoch=1000
for step in range(n_epoch): 
    optimizer.zero_grad()  

    t=torch.rand(v1.shape[0], device=device)

    path_sample = path.sample(t=t, x_0=v1, x_1=v2)
    v_t=flowB(path_sample.x_t, path_sample.t)
    loss = torch.mean((v_t - path_sample.dx_t) ** 2)
    
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

print(torch.mean((flowB.sample(noise=v1,num_steps=64) - v2) ** 2))
if PLOT:
    start_end_subplot(v2.cpu(), flowB.sample(noise=v1,num_steps=64) .detach().cpu(),show=True)